# FUSE Stage 1 — VGGT point-cloud reconstruction

This notebook is the minimal Stage 1 (Point cloud reconstruction of the broken object) path:

```text
video → evenly spaced frames → GPU background removal
      → VGGT cameras + depth → foreground-only point fusion
      → conservative voxel fusion + normals → interactive 3D preview
```

The first and last video frames are not sampled. Foreground masks and point colours are passed through VGGT's own image preprocessing so they remain pixel-aligned with the predicted depth maps.

Main outputs:

```text
data/vggt_outputs/<scene>/raw_vggt_cloud.ply
data/vggt_outputs/<scene>/broken_clean_normals.ply
data/vggt_outputs/<scene>/broken_clean.html
# data/visualizations/<scene>/broken_clean.html
```

The container includes the dependencies for VGGT's `demo_colmap.py` bundle-adjustment path. 
Bundle adjustment is intentionally not run here: its refined cameras must be used in a later dense depth re-fusion step before they can improve this point cloud.


In [ ]:
from __future__ import annotations

import gc
import json
import os
from pathlib import Path

import cv2
import numpy as np
import onnxruntime as ort
import open3d as o3d
import plotly.graph_objects as go
import torch
from IPython.display import display
from PIL import Image
from rembg import new_session, remove
from tqdm.auto import tqdm
from huggingface_hub import hf_hub_download

from vggt_omega.models import VGGTOmega
from vggt_omega.utils.load_fn import load_and_preprocess_images
from vggt_omega.utils.pose_enc import encoding_to_camera

from vggt.utils.geometry import unproject_depth_map_to_point_map
from vggt.utils.pose_enc import pose_encoding_to_extri_intri

if not torch.cuda.is_available():
    raise RuntimeError("A CUDA GPU is required by this notebook.")

DEVICE = "cuda"
GPU_MAJOR, _ = torch.cuda.get_device_capability()
AMP_DTYPE = torch.bfloat16 if GPU_MAJOR >= 8 else torch.float16

torch.set_float32_matmul_precision("high")
gc.collect()
torch.cuda.empty_cache()

print("GPU:", torch.cuda.get_device_name(0))
print("Autocast dtype:", AMP_DTYPE)


In [ ]:
# -------------------------
# USER CONFIGURATION
# -------------------------

CHECKPOINT_PATH = Path(os.environ.get(
    "VGGT_OMEGA_CHECKPOINT",
    "/workspace/.cache/checkpoints/vggt_omega_1b_512.pt",
))
IMAGE_RESOLUTION = 512          # matches the released VGGT-Omega-1B-512 checkpoint
PREPROCESS_MODE = "balanced"    # "balanced" (default) or "max_size" for lower VRAM

N_EXTRACTED_FRAMES =  60 # 24          # sampled from one full turntable revolution
N_VGGT_FRAMES = 16 # 12               # reduce to 10 if GPU memory is insufficient
BAD_FRAME_INDICES = []            # indices in the extracted-frame sequence

OVERWRITE_EXTRACTED_FRAMES = True
OVERWRITE_FOREGROUND = True

CONF_QUANTILE = 0.05              # discard only the globally weakest 5%
VOXELS_PER_ROBUST_DIAGONAL = 5000 #1200 # larger value preserves finer geometry
PLOT_MAX_POINTS = 500_000  # 120_000       # display-only sampling; never changes saved geometry


def find_project_root(start: Path) -> Path:
    start = start.resolve()
    for candidate in [start, *start.parents]:
        if (candidate / "data").is_dir():
            return candidate
    raise FileNotFoundError("Could not find the FUSE root containing data/.")


FUSE_ROOT = find_project_root(Path.cwd())
SCENE_DIR = FUSE_ROOT / "data" / "scenes" 
VIDEO_PATH = SCENE_DIR / "raw_video" / "video.mp4"
EXTRACTED_DIR = SCENE_DIR / "extracted_frames"
FOREGROUND_RGB_DIR = SCENE_DIR / "foreground_rgb"
FOREGROUND_MASK_DIR = SCENE_DIR / "foreground_masks"

VGGT_OUT_DIR = FUSE_ROOT / "data" / "vggt_outputs" 

for directory in [
    EXTRACTED_DIR,
    FOREGROUND_RGB_DIR,
    FOREGROUND_MASK_DIR,
    VGGT_OUT_DIR,
]:
    directory.mkdir(parents=True, exist_ok=True)

RAW_CLOUD_PATH = VGGT_OUT_DIR / "raw_vggt_cloud.ply"
CLEAN_NORMALS_PATH = VGGT_OUT_DIR / "broken_clean_normals.ply" # CLEAN_DIR / "broken_clean_normals.ply"
HTML_PATH = VGGT_OUT_DIR / "broken_clean.html" # VIS_DIR / "broken_clean.html"

print("FUSE root:", FUSE_ROOT)
print("Video:", VIDEO_PATH)
print("Raw cloud:", RAW_CLOUD_PATH)
print("Stage 2 cloud:", CLEAN_NORMALS_PATH)


In [ ]:
# a download-or-fail block for the VGGT-Omega checkpoint, which is not publicly available

if not CHECKPOINT_PATH.exists():
    hf_token = os.environ.get("HF_TOKEN")
    if not hf_token:
        raise RuntimeError(
            "No local VGGT-Omega checkpoint and no HF_TOKEN set. Request "
            "access at https://huggingface.co/facebook/VGGT-Omega, then run "
            "configure-env.sh in FUSE/containers/gpu."
        )
    CHECKPOINT_PATH.parent.mkdir(parents=True, exist_ok=True)
    downloaded = hf_hub_download(
        repo_id="facebook/VGGT-Omega",
        filename="vggt_omega_1b_512.pt",
        token=hf_token,
        local_dir=CHECKPOINT_PATH.parent,
    )
    Path(downloaded).rename(CHECKPOINT_PATH)

print("Checkpoint:", CHECKPOINT_PATH)

In [ ]:
def extract_evenly_spaced_frames(
    video_path: Path,
    output_dir: Path,
    count: int,
    overwrite: bool,
) -> list[Path]:
    # Do not sample the duplicate end pose of a full revolution.
    expected = [output_dir / f"frame_{i:03d}.jpg" for i in range(count)]
    if not overwrite and all(path.exists() for path in expected):
        return expected

    if not video_path.is_file():
        raise FileNotFoundError(video_path)

    if overwrite:
        for path in output_dir.glob("frame_*.jpg"):
            path.unlink()

    capture = cv2.VideoCapture(str(video_path))
    if not capture.isOpened():
        raise RuntimeError(f"Could not open video: {video_path}")

    total = int(capture.get(cv2.CAP_PROP_FRAME_COUNT))
    if total <= 0:
        capture.release()
        raise RuntimeError(f"Could not read frame count: {video_path}")

    count = min(count, total)
    frame_ids = np.linspace(0, total, count, endpoint=False, dtype=int)
    paths: list[Path] = []

    for output_index, frame_id in enumerate(tqdm(frame_ids, desc="Extracting frames")):
        capture.set(cv2.CAP_PROP_POS_FRAMES, int(frame_id))
        ok, frame_bgr = capture.read()
        if not ok:
            capture.release()
            raise RuntimeError(f"Could not read video frame {frame_id}")

        output_path = output_dir / f"frame_{output_index:03d}.jpg"
        if not cv2.imwrite(str(output_path), frame_bgr):
            capture.release()
            raise RuntimeError(f"Could not save {output_path}")
        paths.append(output_path)

    capture.release()
    return paths


frame_paths = extract_evenly_spaced_frames(
    VIDEO_PATH,
    EXTRACTED_DIR,
    N_EXTRACTED_FRAMES,
    OVERWRITE_EXTRACTED_FRAMES,
)

print(f"Extracted {len(frame_paths)} frames")


In [ ]:
def create_foreground_images(
    source_paths: list[Path],
    rgb_dir: Path,
    mask_dir: Path,
    overwrite: bool,
) -> tuple[list[Path], list[Path]]:
    # Save black-composited VGGT inputs and separate alpha masks.
    rgb_paths = [rgb_dir / f"foreground_{i:03d}.png" for i in range(len(source_paths))]
    mask_paths = [mask_dir / f"mask_{i:03d}.png" for i in range(len(source_paths))]

    if not overwrite and all(path.exists() for path in [*rgb_paths, *mask_paths]):
        return rgb_paths, mask_paths

    if "CUDAExecutionProvider" not in ort.get_available_providers():
        raise RuntimeError(
            "ONNX Runtime cannot access CUDA. Check onnxruntime-gpu and the container GPU setup."
        )

    model_dir = Path(os.environ.get("U2NET_HOME", "/workspace/.cache/rembg"))
    model_dir.mkdir(parents=True, exist_ok=True)
    os.environ["U2NET_HOME"] = str(model_dir)

    session = new_session(model_name="u2net", providers=["CUDAExecutionProvider"])

    for source, rgb_path, mask_path in tqdm(
        list(zip(source_paths, rgb_paths, mask_paths)),
        desc="Removing background",
    ):
        if not overwrite and rgb_path.exists() and mask_path.exists():
            continue

        with Image.open(source) as image:
            rgba = remove(image.convert("RGB"), session=session).convert("RGBA")

        alpha = rgba.getchannel("A")
        black_rgb = Image.new("RGB", rgba.size, (0, 0, 0))
        black_rgb.paste(rgba.convert("RGB"), mask=alpha)

        black_rgb.save(rgb_path)
        alpha.save(mask_path)

    del session
    gc.collect()
    return rgb_paths, mask_paths


foreground_paths, foreground_mask_paths = create_foreground_images(
    frame_paths,
    FOREGROUND_RGB_DIR,
    FOREGROUND_MASK_DIR,
    OVERWRITE_FOREGROUND,
)

pairs = [
    (image_path, mask_path)
    for index, (image_path, mask_path) in enumerate(zip(foreground_paths, foreground_mask_paths))
    if index not in set(BAD_FRAME_INDICES)
]

if len(pairs) < 2:
    raise RuntimeError("VGGT requires at least two usable frames.")

if len(pairs) > N_VGGT_FRAMES:
    selected_indices = np.linspace(
        0,
        len(pairs),
        N_VGGT_FRAMES,
        endpoint=False,
        dtype=int,
    )
    pairs = [pairs[index] for index in selected_indices]

vggt_image_paths = [pair[0] for pair in pairs]
vggt_mask_paths = [pair[1] for pair in pairs]

print(f"Selected {len(vggt_image_paths)} VGGT frames:")
for path in vggt_image_paths:
    print(" ", path.name)


In [ ]:
def to_numpy(value):
    if torch.is_tensor(value):
        return value.detach().float().cpu().numpy()
    return np.asarray(value)


def load_omega_model(checkpoint_path: Path) -> VGGTOmega:
    model = VGGTOmega().to(DEVICE).eval()
    state_dict = torch.load(checkpoint_path, map_location="cpu")
    model.load_state_dict(state_dict)
    return model

def preprocess_masks_like_omega(mask_paths: list[Path]) -> np.ndarray:
    mask_tensor = load_and_preprocess_images(
        [str(p) for p in mask_paths],
        image_resolution=IMAGE_RESOLUTION,
        mode=PREPROCESS_MODE,
    )
    mask_score = mask_tensor.mean(dim=1)
    flat = mask_score.flatten(1)
    low = flat.min(dim=1).values[:, None, None]
    high = flat.max(dim=1).values[:, None, None]
    if torch.any(high <= low):
        raise RuntimeError("At least one foreground mask is empty or constant.")
    return (mask_score > (low + high) / 2).cpu().numpy()


def unproject_depth(depth: np.ndarray, extrinsic: np.ndarray, intrinsic: np.ndarray) -> np.ndarray:
    """Manual pinhole unprojection: depth (S,H,W[,1]), intrinsic (S,3,3),
    extrinsic (S,3,4) world-to-camera. Written by hand rather than relying on
    an unconfirmed vggt_omega.utils.geometry helper — swap this out if/when
    you verify the official utility exists and matches this convention."""
    depth = np.asarray(depth)
    if depth.ndim == 4 and depth.shape[-1] == 1:
        depth = depth[..., 0]
    S, H, W = depth.shape
    ys, xs = np.meshgrid(np.arange(H), np.arange(W), indexing="ij")
    world_points = np.zeros((S, H, W, 3), dtype=np.float32)
    for s in range(S):
        fx, fy = intrinsic[s, 0, 0], intrinsic[s, 1, 1]
        cx, cy = intrinsic[s, 0, 2], intrinsic[s, 1, 2]
        z = depth[s]
        x = (xs - cx) * z / fx
        y = (ys - cy) * z / fy
        cam_points = np.stack([x, y, z], axis=-1).reshape(-1, 3)
        R, t = extrinsic[s, :, :3], extrinsic[s, :, 3]
        # extrinsic is world-to-camera; invert to place points in world space
        world = (cam_points - t) @ R
        world_points[s] = world.reshape(H, W, 3)
    return world_points


def run_vggt(image_paths: list[Path], mask_paths: list[Path]) -> dict:
    image_tensor_cpu = load_and_preprocess_images(
        [str(p) for p in image_paths],
        image_resolution=IMAGE_RESOLUTION,
        mode=PREPROCESS_MODE,
    )
    if image_tensor_cpu.min() < -1e-4 or image_tensor_cpu.max() > 1.0001:
        raise RuntimeError("Unexpected VGGT image range; cannot export aligned RGB safely.")

    aligned_rgb = (
        image_tensor_cpu.permute(0, 2, 3, 1).clamp(0, 1).mul(255).round().to(torch.uint8).numpy()
    )
    aligned_masks = preprocess_masks_like_omega(mask_paths)
    if aligned_rgb.shape[:3] != aligned_masks.shape:
        raise RuntimeError("Preprocessed RGB and foreground-mask shapes do not match.")

    images = image_tensor_cpu.to(DEVICE)
    del image_tensor_cpu

    model = load_omega_model(CHECKPOINT_PATH)

    with torch.inference_mode(), torch.amp.autocast("cuda", dtype=AMP_DTYPE):
        predictions = model(images[None])

    extrinsic, intrinsic = encoding_to_camera(
        predictions["pose_enc"], predictions["images"].shape[-2:],
    )
    depth_map = predictions["depth"]
    depth_conf = predictions["depth_conf"]
    world_points = unproject_depth(
        to_numpy(depth_map.squeeze(0)),
        to_numpy(extrinsic.squeeze(0)),
        to_numpy(intrinsic.squeeze(0)),
    )

    result = {
        "image_paths": [str(p) for p in image_paths],
        "mask_paths": [str(p) for p in mask_paths],
        "extrinsic": to_numpy(extrinsic.squeeze(0)),
        "intrinsic": to_numpy(intrinsic.squeeze(0)),
        "depth_map": to_numpy(depth_map.squeeze(0)),
        "depth_conf": to_numpy(depth_conf.squeeze(0)),
        "world_points": world_points,
        "aligned_rgb": aligned_rgb,
        "aligned_masks": aligned_masks,
    }

    del model, images, predictions, extrinsic, intrinsic, depth_map, depth_conf
    gc.collect()
    torch.cuda.empty_cache()
    return result


def run_vggt(image_paths: list[Path], mask_paths: list[Path]) -> dict:
    image_tensor_cpu = load_and_preprocess_images([str(path) for path in image_paths])
    if image_tensor_cpu.min() < -1e-4 or image_tensor_cpu.max() > 1.0001:
        raise RuntimeError("Unexpected VGGT image range; cannot export aligned RGB safely.")

    aligned_rgb = (
        image_tensor_cpu.permute(0, 2, 3, 1)
        .clamp(0, 1)
        .mul(255)
        .round()
        .to(torch.uint8)
        .numpy()
    )
    aligned_masks = preprocess_masks_like_omega(mask_paths)

    if aligned_rgb.shape[:3] != aligned_masks.shape:
        raise RuntimeError("Preprocessed RGB and foreground-mask shapes do not match.")

    images = image_tensor_cpu.to(DEVICE)
    del image_tensor_cpu

    print("VGGT tensor:", tuple(images.shape))

    with torch.inference_mode(), torch.amp.autocast("cuda", dtype=AMP_DTYPE):
        images_batched = images[None]
        aggregated_tokens, patch_start = model.aggregator(images_batched)

        pose_encoding = model.camera_head(aggregated_tokens)[-1]
        extrinsic, intrinsic = pose_encoding_to_extri_intri(
            pose_encoding,
            images_batched.shape[-2:],
        )

        depth_map, depth_conf = model.depth_head(
            aggregated_tokens,
            images_batched,
            patch_start,
        )

        world_points = unproject_depth_map_to_point_map(
            depth_map.squeeze(0),
            extrinsic.squeeze(0),
            intrinsic.squeeze(0),
        )

    result = {
        "image_paths": [str(path) for path in image_paths],
        "mask_paths": [str(path) for path in mask_paths],
        "extrinsic": to_numpy(extrinsic.squeeze(0)),
        "intrinsic": to_numpy(intrinsic.squeeze(0)),
        "depth_map": to_numpy(depth_map.squeeze(0)),
        "depth_conf": to_numpy(depth_conf.squeeze(0)),
        "world_points": to_numpy(world_points),
        "aligned_rgb": aligned_rgb,
        "aligned_masks": aligned_masks,
    }

    del model, images, images_batched, aggregated_tokens, pose_encoding
    del extrinsic, intrinsic, depth_map, depth_conf, world_points
    gc.collect()
    torch.cuda.empty_cache()
    return result


vggt_result = run_vggt(vggt_image_paths, vggt_mask_paths)

np.save(VGGT_OUT_DIR / "cameras_extrinsic.npy", vggt_result["extrinsic"])
np.save(VGGT_OUT_DIR / "cameras_intrinsic.npy", vggt_result["intrinsic"])
np.save(VGGT_OUT_DIR / "depth_map.npy", vggt_result["depth_map"])
np.save(VGGT_OUT_DIR / "depth_conf.npy", vggt_result["depth_conf"])
np.save(VGGT_OUT_DIR / "foreground_masks.npy", vggt_result["aligned_masks"])

with open(VGGT_OUT_DIR / "selected_inputs.json", "w", encoding="utf-8") as file:
    json.dump(
        {
            "images": vggt_result["image_paths"],
            "masks": vggt_result["mask_paths"],
        },
        file,
        indent=2,
    )

print("Saved VGGT cameras, depth, confidence, and aligned masks to", VGGT_OUT_DIR)


In [ ]:
def squeeze_confidence(confidence: np.ndarray) -> np.ndarray:
    confidence = np.asarray(confidence)
    if confidence.ndim == 4 and confidence.shape[-1] == 1:
        confidence = confidence[..., 0]
    return confidence


def make_point_cloud(points: np.ndarray, colors: np.ndarray) -> o3d.geometry.PointCloud:
    cloud = o3d.geometry.PointCloud()
    cloud.points = o3d.utility.Vector3dVector(points.astype(np.float64))
    cloud.colors = o3d.utility.Vector3dVector(colors.astype(np.float64) / 255.0)
    return cloud


def fuse_foreground_points(prediction: dict, confidence_quantile: float):
    if not 0 <= confidence_quantile < 1:
        raise ValueError("CONF_QUANTILE must be in [0, 1).")

    points = np.asarray(prediction["world_points"], dtype=np.float32)
    confidence = squeeze_confidence(prediction["depth_conf"])
    colors = np.asarray(prediction["aligned_rgb"], dtype=np.uint8)
    masks = np.asarray(prediction["aligned_masks"], dtype=bool)

    expected = points.shape[:3]
    if confidence.shape != expected or colors.shape[:3] != expected or masks.shape != expected:
        raise RuntimeError(
            f"Prediction alignment failure: points={expected}, confidence={confidence.shape}, "
            f"colors={colors.shape[:3]}, masks={masks.shape}"
        )

    valid = masks & np.isfinite(confidence) & np.isfinite(points).all(axis=-1)
    if not np.any(valid):
        raise RuntimeError("No finite foreground points were produced.")

    threshold = float(np.quantile(confidence[valid], confidence_quantile))
    valid &= confidence >= threshold

    point_parts = []
    color_parts = []
    for view_index in range(points.shape[0]):
        point_parts.append(points[view_index][valid[view_index]])
        color_parts.append(colors[view_index][valid[view_index]])
        print(f"View {view_index:02d}: {valid[view_index].sum():,} points")

    fused_points = np.concatenate(point_parts, axis=0)
    fused_colors = np.concatenate(color_parts, axis=0)
    print(f"Global confidence threshold: {threshold:.4f}")
    print(f"Fused foreground cloud: {len(fused_points):,} points")
    return fused_points, fused_colors


raw_points, raw_colors = fuse_foreground_points(vggt_result, CONF_QUANTILE)
raw_cloud = make_point_cloud(raw_points, raw_colors)

if not o3d.io.write_point_cloud(
    str(RAW_CLOUD_PATH),
    raw_cloud,
    write_ascii=False,
    compressed=True,
):
    raise RuntimeError(f"Could not save {RAW_CLOUD_PATH}")

print("Saved raw cloud:", RAW_CLOUD_PATH)


In [ ]:
def median_nearest_neighbor_distance(
    cloud: o3d.geometry.PointCloud,
    sample_size: int = 20_000,
    seed: int = 0,
) -> float:
    points = np.asarray(cloud.points)
    if len(points) < 2:
        raise RuntimeError("Point cloud is too small for normal estimation.")

    rng = np.random.default_rng(seed)
    indices = (
        rng.choice(len(points), sample_size, replace=False)
        if len(points) > sample_size
        else np.arange(len(points))
    )

    tree = o3d.geometry.KDTreeFlann(cloud)
    distances = []
    for index in indices:
        _, _, squared_distances = tree.search_knn_vector_3d(cloud.points[index], 2)
        if len(squared_distances) == 2:
            distances.append(np.sqrt(squared_distances[1]))

    if not distances:
        raise RuntimeError("Could not estimate point spacing.")
    return float(np.median(distances))


# Robust percentiles determine only the voxel scale; they do not delete extremities.
lower = np.percentile(raw_points, 0.5, axis=0)
upper = np.percentile(raw_points, 99.5, axis=0)
robust_diagonal = float(np.linalg.norm(upper - lower))
if not np.isfinite(robust_diagonal) or robust_diagonal <= 0:
    raise RuntimeError("Invalid point-cloud scale.")

voxel_size = robust_diagonal / VOXELS_PER_ROBUST_DIAGONAL
clean_cloud = raw_cloud.voxel_down_sample(voxel_size)

# No statistical/radius outlier filter is applied: thin tails, horns, fracture rims,
# and other valid low-density geometry must not be deleted automatically.
median_spacing = median_nearest_neighbor_distance(clean_cloud)
normal_radius = max(4.0 * voxel_size, 6.0 * median_spacing)

clean_cloud.estimate_normals(
    search_param=o3d.geometry.KDTreeSearchParamHybrid(
        radius=normal_radius,
        max_nn=50,
    )
)
clean_cloud.orient_normals_consistent_tangent_plane(30)

if not o3d.io.write_point_cloud(
    str(CLEAN_NORMALS_PATH),
    clean_cloud,
    write_ascii=False,
    compressed=True,
):
    raise RuntimeError(f"Could not save {CLEAN_NORMALS_PATH}")

print("Raw points:", len(raw_cloud.points))
print("Voxel-fused points:", len(clean_cloud.points))
print("Voxel size:", voxel_size)
print("Normal radius:", normal_radius)
print("Saved Stage 2 cloud:", CLEAN_NORMALS_PATH)


In [ ]:
def show_point_cloud(
    cloud_path: Path,
    html_path: Path,
    max_points: int = 300_000,
    point_size: float = 1.5,
):
    cloud = o3d.io.read_point_cloud(str(cloud_path))
    points = np.asarray(cloud.points)
    if len(points) == 0:
        raise RuntimeError(f"Empty point cloud: {cloud_path}")

    colors = (
        np.asarray(cloud.colors)
        if cloud.has_colors()
        else np.full((len(points), 3), 0.65)
    )

    if len(points) > max_points:
        rng = np.random.default_rng(0)
        indices = rng.choice(len(points), max_points, replace=False)
        points = points[indices]
        colors = colors[indices]

    rgb = (np.clip(colors, 0, 1) * 255).astype(np.uint8)
    color_strings = [f"rgb({r},{g},{b})" for r, g, b in rgb]

    figure = go.Figure(
        go.Scatter3d(
            x=points[:, 0],
            y=points[:, 1],
            z=points[:, 2],
            mode="markers",
            marker={"size": point_size, "color": color_strings, "opacity": 0.95},
        )
    )
    figure.update_layout(
        title=f"FUSE VGGT cloud",
        scene={"aspectmode": "data"},
        width=950,
        height=750,
        margin={"l": 0, "r": 0, "b": 0, "t": 40},
    )

    html_path.parent.mkdir(parents=True, exist_ok=True)
    figure.write_html(str(html_path), include_plotlyjs=True, full_html=True)
    display(figure)
    print("Saved interactive visualization:", html_path)
    return figure


figure = show_point_cloud(
    CLEAN_NORMALS_PATH,
    HTML_PATH,
    max_points=PLOT_MAX_POINTS,
)


## Result

Use this point cloud as the measured geometry for Stage 2:

```text
data/cleaned_geometry/<scene>/broken_clean_normals.ply
```

The Plotly display samples points only for rendering. It does not alter the saved PLY.

If the saved cloud still contains offset shells after this corrected fusion, the remaining problem is camera-pose consistency rather than point filtering. The next step is VGGT track prediction + shared-camera COLMAP bundle adjustment, followed by dense depth re-unprojection with the refined cameras.
